# Relaxation and contact settings

Relaxation is a quasi-static sequence of contact, stretch, rest-bend,
and admissible-curvature projections. Convergence requires both the
penetration and curvature residuals to satisfy their configured limits.
`RelaxationSettings` is run-wide: it applies to every step of a recipe
unless a step receives temporary `RelaxationOverrides` (tutorial 09).

In [ ]:
# Settings objects expose the same defaults used by Rust.
import tangle
from tangle.units import um

## Every `RelaxationSettings` field

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `backend` | CubeCL execution backend. | `"wgpu"` or `"cpu"` |
| `motion_model` | How contact corrections move a fiber. | `"flexible"` or `"rigid_translation"` |
| `pin_fiber_ends` | Prevents the first and last vertex from moving. | boolean |
| `penetration_tolerance` | Largest accepted capsule overlap. | length, m |
| `force_full_iterations` | Runs the entire iteration budget instead of accepting early. | boolean |
| `correction_fraction` | Fraction of each contact correction applied per pass. | dimensionless |
| `contact_aggregation` | Combines multiple corrections at a vertex. | `uniform_average`, `penetration_weighted`, `deepest_only` |
| `stretch_stiffness` | Rest-length projection strength. | dimensionless |
| `bend_stiffness` | Rest-shape bending projection strength. | dimensionless |
| `curvature_limit_stiffness` | Admissible-curvature projection strength. | dimensionless |
| `curvature_limit_safety_margin` | Keeps projected bends inside the hard limit. | ratio |
| `curvature_ratio_tolerance` | Excess ratio allowed above one: the accepted curvature ratio is at most 1 + this tolerance. | dimensionless excess |
| `constraint_iterations` | Constraint sweeps in each solver iteration. | count |
| `curvature_cleanup_sweeps` | Extra hard-curvature projections per iteration. | count |
| `max_step` | Maximum vertex displacement in one correction. | length, m |
| `max_iterations` | Global relaxation iteration budget. | count |
| `iterations_per_batch` | Iterations in one scheduler/device batch. | count |
| `debug_snapshot_interval` | Periodic OVITO cadence; `None` keeps only keyframes when run() receives a debug path. | iterations or `None` |
| `save_assembled_reference` | Stores the converged geometry as an assembled reference. | boolean |
| `cell_size_scale` | Broad-phase cell size relative to the smallest admissible cell. | at least 1 |
| `neighbor_skin_scale` | Extra Verlet neighbor-list search distance, as a multiple of the largest fiber radius. | finite, at least 0 |
| `neighbor_capacity` | Neighbor-list length per segment; overflowing segments fall back to scanning cells. | count, at least 1 |
| `adaptive_segmentation` | Optional adaptive refinement configuration. | `AdaptiveSegmentationSettings` or `None` |

In [ ]:
# Read defaults from the compiled extension instead of duplicating
# them in documentation that could become stale.
settings = tangle.RelaxationSettings()
fields = ['backend', 'motion_model', 'pin_fiber_ends', 'penetration_tolerance', 'force_full_iterations', 'correction_fraction', 'contact_aggregation', 'stretch_stiffness', 'bend_stiffness', 'curvature_limit_stiffness', 'curvature_limit_safety_margin', 'curvature_ratio_tolerance', 'constraint_iterations', 'curvature_cleanup_sweeps', 'max_step', 'max_iterations', 'iterations_per_batch', 'debug_snapshot_interval', 'save_assembled_reference', 'cell_size_scale', 'neighbor_skin_scale', 'neighbor_capacity', 'adaptive_segmentation']
{name: getattr(settings, name) for name in fields}

## Building and editing settings

Pass only the values that differ from the defaults as keyword
arguments. `replace(**changes)` returns a modified copy and leaves the
original alone; attributes remain assignable for incremental edits.
Misspelled keywords raise `TypeError` and unknown option strings raise
`ValueError` on the line that introduced them.

In [ ]:
# Keyword construction names only what differs from the defaults.
settings = tangle.RelaxationSettings(
    backend="cpu", max_iterations=500, penetration_tolerance=0.1 * um
)
# replace() derives a stricter variant without touching `settings`.
strict = settings.replace(max_iterations=5_000, curvature_ratio_tolerance=0.0)
# Direct assignment still works for one-off edits.
strict.max_step = 2 * um

# Typos and invalid option strings fail immediately.
for bad in ({"backnd": "cpu"}, {"backend": "cuda"}):
    try:
        tangle.RelaxationSettings(**bad)
    except (TypeError, ValueError) as error:
        print(type(error).__name__, error)
settings.max_iterations, strict.max_iterations

## Cell-list broad phase

`cell_size_scale` sets the broad-phase cell size. A value of 1
uses the minimum admissible cell size; larger cells reduce cell count
but increase candidates per cell. It must be at least 1.

Each segment also keeps a Verlet neighbor list built from the cells.
`neighbor_skin_scale` is the extra search distance, as a multiple of
the largest fiber radius. Lists are rebuilt only after some vertex
moves more than half the skin, so a larger skin means fewer rebuilds
but longer lists. `neighbor_capacity` is the list length per segment;
a segment with more neighbors falls back to scanning its cells, so the
result is the same, only slower. The skin must be finite and at least 0,
and the capacity at least 1.

In [ ]:
# Broad-phase cells must be at least one contact diameter wide. Larger
# values trade fewer cells for more candidate capsule pairs per cell.
coarse_cells = settings.replace(
    cell_size_scale=1.5, neighbor_skin_scale=2.0, neighbor_capacity=48
)
coarse_cells.to_dict()

Use `motion_model="rigid_translation"` for straight rigid fibers and
`"flexible"` for vertex-level deformation. The rigid mode also preserves
curved centerlines; it applies translation only, not rotation.
`backend="wgpu"` is the
accelerator path; `"cpu"` runs the same CubeCL kernels through the
native multithreaded CPU runtime.